# RAD-DINO layer cut — how much accuracy is left after *k* of 12 blocks? (official split, seed 42)
Protocol: `evaluation_protocol.yaml` **v2.1.0**, section `layer_depth` (single seed, no TTA).

The cascade saved little compute on the official test set (only 38.5 % of test images are normal, so a gate cannot skip many).
This notebook asks a different question: **if RAD-DINO stops after block *k* on every image, how much accuracy is lost and how much time is saved?**
No gate is involved, so the validation → test shift that broke the cascade does not apply.

```
X-ray → RAD-DINO blocks 1 … k → final layernorm → CLS ⊕ mean(patch) → linear head (one per k) → 14 findings
```

* Source model: the **fine-tuned seed-42 RAD-DINO** (`M09_RAD_DINO_518_CR`, EMA weights, `checkpoint_best.pt`, SHA-256 `070f90c7…`). It is not retrained.
* For each depth *k* = 1 … 12 a linear head (a *probe*) is trained on train-set features, early-stopped on validation.
* Evaluation is identical to v2.0.0: per-class Platt calibration and F1 thresholds on validation, patient-level bootstrap on test.
* **Selection rule (fixed before any result):** the smallest *k* whose validation macro AUROC is within **0.005** of full RAD-DINO.

### Sessions (Kaggle)
| Session | Flags | Accelerator | Time | Attach |
|---|---|---|---|---|
| **L** | `RUN_LAYERS=True` | GPU T4, Internet ON | ~2.5 h | NIH dataset (`nih-chest-xrays/data`); the **session-B output** that holds `checkpoint_best.pt`; `results_s42` (split file, M08/M09 logits, timing) |
| **E** | `RUN_EVAL=True` | CPU is enough | ~45 min (1,000 bootstrap resamples × 14 systems) | session-L output + `results_s42` |

* Both flags can be `True` in one GPU session (L first, then E), but E then spends ~45 min of GPU quota on CPU work; running E as a separate CPU session saves it. Test labels are used only in E, after every choice is frozen.
* **Step 0 checks the checkpoint first** (about 1 minute). If it is missing, the session stops before any GPU work.
* L resumes: features, probe logits and timing files that already exist in the output are not recomputed.
* Output: `features/` (~4.1 GB fp16, kept so probes can be refitted without a GPU), one run folder per depth, `timing_*_Lkk.json`, T7–T9, `layer_depth_curve.png`, `qc_report_layers.csv`.

In [ ]:
# ============================ CONFIG ============================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # determinism; set before CUDA starts
import json, glob, time, math, random, hashlib, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
from IPython.display import display
warnings.filterwarnings("ignore")

# ---- what to run in THIS session ----
RUN_LAYERS = True       # Session L (GPU): checkpoint check, layer features, probes, timing
RUN_EVAL   = True       # Session E (CPU ok): calibration, thresholds, depth selection, test metrics, tables
DEBUG      = False      # small subsets, 2 probe epochs, short timing; folders get a _debug tag and must never be reported

SEED = 42
PROTOCOL_VERSION = "2.1.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      # NIH images + Data_Entry_2017.csv (session L)
NUM_WORKERS = int(os.environ.get("CXR_NUM_WORKERS", 4))

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
EXPECTED_COUNTS = {"train": 73916, "val": 12608, "test": 25596}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 runs

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)
ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)

# ---- source models (seed-42 runs of protocol v2.0.0) ----
CNN_ID = "M08_CONVNEXT_TINY_ASL_384_CR"
M09_ID = "M09_RAD_DINO_518_CR"
EXPECTED_M09_CKPT_SHA256 = "070f90c7cf44c54447a015791f5a18d0eb0d8b05bce070450b6c35e91008bd93"   # from the run's config.json
VIT_CFG = dict(kind="raddino", model_id=M09_ID, hf_name="microsoft/rad-dino",
               hf_revision="110cbc18d5133582e320b43d53bf5c44e410c936",    # commit used by the seed-42 run
               img_size=518, mean=(0.5307,) * 3, std=(0.2583,) * 3, batch_size=32)

# ---- layer-depth experiment (fixed before any probe result) ----
DEPTHS = list(range(1, 13))
DEPTH_TOLERANCE = 0.005        # smallest depth with val macro AUROC >= RAD-DINO alone - 0.005
PROBE = dict(lr=1e-3, weight_decay=1e-4, batch_size=1024, max_epochs=50, patience=5)
KEEP_FEATURES = True           # ~4.1 GB fp16; lets probes be refitted without a GPU
QC_L3 = dict(min_corr=0.999, max_auroc_diff=0.001)    # L12 features + trained head must reproduce the saved M09 logits

# ---- evaluation settings (same as v2.0.0) ----
N_BOOT, BOOTSTRAP_SEED, ALPHA = 1000, 20260922, 0.05
ECE_BINS = 15
MIN_SUPPORT_WARN = 100
LATENCY_BATCH_SIZES = [1, 32]
WARMUP_ITERS, TIMED_ITERS, TIMING_REPEATS = 50, 200, 3

RUN_TAG = DATASET_ID + ("_debug" if DEBUG else "")     # probe outputs only; source runs are always the real ones
if DEBUG:
    PROBE.update(max_epochs=2)
    N_BOOT, WARMUP_ITERS, TIMED_ITERS, TIMING_REPEATS = 50, 3, 5, 1
FEAT_DIR = f"{OUT_DIR}/features{'_debug' if DEBUG else ''}"
os.makedirs(OUT_DIR, exist_ok=True)

def depth_id(k):
    return f"{M09_ID}_L{k:02d}"

# ---------------- helpers (copied from 2-staged_official_split.ipynb) ----------------
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def save_table(df_, name):
    """CSV, UTF-8, full precision (rounding only when displayed)."""
    df_.to_csv(f"{OUT_DIR}/{name}.csv", index=False, float_format="%.17g", na_rep="")

def fmt(x, d=4):
    return "n/a" if x is None or not np.isfinite(x) else f"{np.round(x, d):.{d}f}"

def _walk(bases=(OUT_DIR, INPUT_DIR)):
    """os.walk over outputs and inputs, skipping the 112k-image folders."""
    for base in dict.fromkeys(bases):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith("images")]
            yield root, dirs, files

def find_file(fname):
    for root, _, files in _walk():
        if fname in files:
            return os.path.join(root, fname)
    return None

def run_folder_name(model_id, tag=DATASET_ID):
    return f"{tag}__{SPLIT_ID}__{model_id}__s{SEED}"

def find_run_dir(model_id, tag=DATASET_ID):
    """Seed-42 run folder of a model that contains exported logits (also accepts …__{hash}__s42__r0 names).
    The same folder name attached twice counts once (OUT_DIR is searched first)."""
    hits = {}
    for root, dirs, files in _walk():
        name = os.path.basename(root)
        if ("logits_test.npz" in files and name.startswith(f"{tag}__{SPLIT_ID}__{model_id}__")
                and (name.endswith(f"__s{SEED}") or f"__s{SEED}__" in name)):
            hits.setdefault(name, root)
    assert len(hits) <= 1, f"several seed-{SEED} folders for {model_id}: {sorted(hits)} - attach only one"
    return next(iter(hits.values()), None)

QC = {}
def qc(id_, ok, detail, severity="error"):
    QC[id_] = dict(passed=bool(ok), severity=severity, detail=str(detail))
    print(("✓ " if ok else ("✗ " if severity == "error" else "⚠ ")) + f"{id_}: {detail}")

## 0. Step 0 — find and verify the fine-tuned RAD-DINO checkpoint (session L)
The seed-42 RAD-DINO checkpoint must be attached (session-B output). Its SHA-256 must equal the hash stored in the run's `config.json`.
If it is missing, nothing else in session L runs.

In [ ]:
def find_checkpoint():
    """checkpoint_best.pt inside a seed-42 M09 run folder whose SHA-256 matches the recorded one."""
    hits = [os.path.join(r, "checkpoint_best.pt") for r, _, fs in _walk()
            if "checkpoint_best.pt" in fs and f"__{M09_ID}__" in os.path.basename(r)]
    for p in hits:
        h = sha256_file(p)
        print(f"  found {p} (sha256 {h[:12]}…)")
        if h == EXPECTED_M09_CKPT_SHA256:
            return p, h
    return None, (sha256_file(hits[0]) if hits else None)

if RUN_LAYERS:
    CKPT, ckpt_sha = find_checkpoint()
    qc("QC-L1 checkpoint", CKPT is not None,
       f"{CKPT} sha256 {EXPECTED_M09_CKPT_SHA256[:12]}…" if CKPT else
       f"no checkpoint_best.pt with sha256 {EXPECTED_M09_CKPT_SHA256[:12]}… (found: {ckpt_sha})")
    if CKPT is None:
        raise RuntimeError(
            "STOP: the fine-tuned RAD-DINO checkpoint is not attached.\n"
            "  1. Open the Kaggle notebook version that trained M09 (session B) -> Output tab -> look for checkpoint_best.pt.\n"
            "  2. Attach that output here (Add Input -> Your Work -> Notebook Output) and run again.\n"
            "  If no output version holds it, the checkpoint is lost: do not run session L (a new plan is needed).")
    M09_DIR = find_run_dir(M09_ID)
    assert M09_DIR, "no seed-42 M09 run folder with logits_*.npz found - attach results_s42 or the session-B output"
    print(f"M09 logits: {M09_DIR}")

## 1. Data (session L)
The split comes from the attached `split_NIH_OFFICIAL_PV.csv`; its SHA-256 must equal the split of the seed-42 runs (no re-split, so no dependence on library versions).
Labels are built exactly as in `2-staged_official_split.ipynb`. **QC-L2:** validation/test images and labels must equal the ones stored in M09's logits.

In [ ]:
if RUN_LAYERS:
    split_path = find_file(f"split_{SPLIT_ID}.csv")
    assert split_path, "split_NIH_OFFICIAL_PV.csv not found - attach results_s42 or the session-B output"
    SPLIT_SHA256 = sha256_file(split_path)
    assert SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, f"split file hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}"
    split_df = pd.read_csv(split_path)

    meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
    if not os.path.exists(meta_csv):
        meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
    assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - attach the NIH dataset"
    DATA_ROOT = os.path.dirname(meta_csv)
    png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
          [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
    meta = pd.read_csv(meta_csv)
    labels = meta["Finding Labels"].str.split("|")
    for c in CLASSES:
        meta[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
    df = split_df.merge(meta[["Image Index"] + CLASSES].rename(columns={"Image Index": "image_index"}),
                        on="image_index", how="left", validate="one_to_one")
    df["path"] = df.image_index.map({os.path.basename(p): p for p in png})
    assert df.path.notna().all() and df[CLASSES].notna().all().all(), "images or labels missing"
    df = df.sort_values("image_index").reset_index(drop=True)                  # canonical order
    FRAMES = {s: df[df.partition == s].reset_index(drop=True) for s in ("train", "val", "test")}
    assert DEBUG or {s: len(f) for s, f in FRAMES.items()} == EXPECTED_COUNTS, "partition sizes"
    if DEBUG:
        FRAMES = {s: f.sample(n, random_state=SEED).sort_values("image_index").reset_index(drop=True)
                  for (s, f), n in zip(FRAMES.items(), (2000, 800, 800))}

    def align(frame, npz):
        """Row positions of frame's images inside an exported npz (None if any is missing)."""
        pos = pd.Series(np.arange(len(npz["image_index"])), index=npz["image_index"].astype(str))
        idx = pos.reindex(frame.image_index.values)
        return None if idx.isna().any() else idx.values.astype(int)

    M09_NPZ = {p: np.load(f"{M09_DIR}/logits_{p}.npz") for p in ("val", "test")}
    ok = []
    for p in ("val", "test"):
        idx = align(FRAMES[p], M09_NPZ[p])
        ok.append(idx is not None and (DEBUG or len(idx) == len(M09_NPZ[p]["image_index"]))
                  and np.array_equal(M09_NPZ[p]["labels"][idx], FRAMES[p][CLASSES].values.astype(np.uint8)))
    qc("QC-L2 images and labels", all(ok), f"val/test identical to M09 logits: {ok}")
    assert all(ok), "QC-L2 failed"
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | " + " | ".join(f"{s} {len(f):,}" for s, f in FRAMES.items()))

## 2. Model and data code
`CXRDataset`, `loader`, `AsymmetricLoss` and the helpers are copied unchanged from `2-staged_official_split.ipynb`.
`RadDinoClassifier` is the same architecture (ViT-B/14 + CLS ⊕ mean-patch pooling + dropout/linear head); it takes an optional `depth`
(number of blocks kept) and pins the Hugging Face revision. `layer_features` runs the blocks one by one and pools after every block.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")               # grayscale -> [0, 1] -> 3 channels
        return self.norm(self.tfm(img).repeat(3, 1, 1)), self.y[i]

def loader(frame, cfg, train, batch_size=None):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)

class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021)."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class RadDinoClassifier(nn.Module):
    """RAD-DINO ViT-B/14 + dual pooling (CLS ⊕ mean patch = 1536-d). depth = number of blocks kept (None = all 12).
    Weights are random here: they come from checkpoint_best.pt (probes) or do not matter (timing)."""
    def __init__(self, cfg, depth=None):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        conf = AutoConfig.from_pretrained(cfg["hf_name"], revision=cfg["hf_revision"])
        if depth is not None:
            conf.num_hidden_layers = depth
        self.backbone = AutoModel.from_config(conf)
        self.n_reg = getattr(conf, "num_register_tokens", 0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * conf.hidden_size, C))

    def pool(self, h):
        return torch.cat([h[:, 0], h[:, 1 + self.n_reg:].mean(1)], dim=1)

    def forward(self, x):
        return self.head(self.pool(self.backbone(pixel_values=x).last_hidden_state))

def layer_features(model, x, depths):
    """{k: pooled 1536-d features after block k (final layernorm applied)} for k in depths, in one pass."""
    bb, out = model.backbone, {}
    h = bb.embeddings(x)
    for i, blk in enumerate(bb.encoder.layer, 1):
        h = blk(h)
        h = h[0] if isinstance(h, (tuple, list)) else h
        if i in depths:
            out[i] = model.pool(bb.layernorm(h))
        if i == max(depths):
            break
    return out

def load_m09(path):
    """Fine-tuned M09 (EMA weights). Accepts a plain state dict, a wrapped one, and DataParallel/EMA 'module.' prefixes."""
    sd = torch.load(path, map_location="cpu", weights_only=False)
    for key in ("state_dict", "model", "module"):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            sd = sd[key]
    sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
    model = RadDinoClassifier(VIT_CFG)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"checkpoint keys do not match RadDinoClassifier:\n  missing {missing[:10]}\n  unexpected {unexpected[:10]}")
    return model.to(DEVICE).eval()

## 3. Evaluation module
Copied unchanged from `2-staged_official_split.ipynb` (metrics, Platt calibration, F1 thresholds, Holm).

In [ ]:
HIGHER, LOWER = "higher_is_better", "lower_is_better"

DIRECTION = dict(auroc=HIGHER, auprc=HIGHER, precision=HIGHER, recall=HIGHER, specificity=HIGHER, f1=HIGHER, ece=LOWER,
                 stage2_fraction=LOWER, compute_ratio_vs_raddino=LOWER, labels_missed_by_stage1=LOWER)

THRESHOLD_FREE = {"auroc", "auprc", "ece"}

def auroc_cols(y, s):
    """Mann-Whitney AUROC per class (= sklearn roc_auc_score). No positives or no negatives -> NaN."""
    y = y.astype(bool)
    P = y.sum(0).astype(float); N = len(y) - P
    r = rankdata(s, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        a = ((r * y).sum(0) - P * (P + 1) / 2) / (P * N)
    a[(P == 0) | (N == 0)] = np.nan
    return a

def ap_cols(y, s):
    """Average precision, step-wise (= sklearn average_precision_score)."""
    out = np.full(y.shape[1], np.nan)
    for c in range(y.shape[1]):
        yc = y[:, c].astype(bool); P = yc.sum()
        if P == 0 or P == len(yc):
            continue
        o = np.argsort(-s[:, c], kind="mergesort")
        ss, yy = s[o, c], yc[o]
        last = np.r_[ss[1:] != ss[:-1], True]
        tp = np.cumsum(yy)[last]; n_pred = np.arange(1, len(yy) + 1)[last]
        out[c] = np.sum(np.diff(np.r_[0.0, tp / P]) * (tp / n_pred))
    return out

def ece_cols(y, p, bins=ECE_BINS):
    out = np.zeros(y.shape[1])
    for c in range(y.shape[1]):
        idx = np.minimum((p[:, c] * bins).astype(int), bins - 1)
        out[c] = sum(abs(p[idx == k, c].mean() - y[idx == k, c].mean()) * (idx == k).mean() for k in range(bins) if (idx == k).any())
    return out

def metric_values(y, score, pred, with_ece=False):
    """{(metric, class): value}; class = label name, MACRO (nanmean over classes) or MICRO."""
    y, pred = y.astype(bool), pred.astype(bool)
    P = y.sum(0); N = len(y) - P
    tp = (pred & y).sum(0); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0); tn = (~pred & ~y).sum(0)
    pc = dict(auroc=auroc_cols(y, score), auprc=ap_cols(y, score),
              precision=np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0),           # no positive prediction -> 0
              recall=np.where(P > 0, tp / np.maximum(P, 1), np.nan),
              specificity=np.where(N > 0, tn / np.maximum(N, 1), np.nan),
              f1=np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan))
    if with_ece:
        pc["ece"] = ece_cols(y, score)
    v = {}
    for m, arr in pc.items():
        arr = arr.astype(float)
        v.update({(m, CLASSES[c]): float(arr[c]) for c in range(C)})
        v[(m, "MACRO")] = float(np.nanmean(arr)) if np.isfinite(arr).any() else np.nan
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    v[("precision", "MICRO")] = float(TP / (TP + FP)) if TP + FP else 0.0
    v[("recall", "MICRO")] = float(TP / (TP + FN)) if TP + FN else np.nan
    v[("f1", "MICRO")] = float(2 * TP / (2 * TP + FP + FN)) if 2 * TP + FP + FN else np.nan
    return v

def fit_platt(z, y):
    """p = sigmoid(a_c * logit + b_c), per class, binary NLL, L-BFGS."""
    a, b = np.ones(z.shape[1]), np.zeros(z.shape[1])
    for c in range(z.shape[1]):
        zc, yc = z[:, c].astype(np.float64), y[:, c].astype(np.float64)
        def f(w):
            u = w[0] * zc + w[1]
            g = expit(u) - yc
            return np.mean(np.logaddexp(0, u) - yc * u), np.array([np.mean(g * zc), np.mean(g)])
        a[c], b[c] = minimize(f, [1.0, 0.0], jac=True, method="L-BFGS-B", bounds=[(1e-3, 100), (-50, 50)],
                              options=dict(maxiter=1000)).x
    return a, b

def best_f1_threshold(y, s, tp0=0, fp0=0, fn0=0):
    """Threshold on s maximising F1, given decisions already fixed elsewhere (tp0, fp0, fn0).
    Candidates = unique scores; ties -> smallest threshold. Returns (threshold, f1)."""
    y = y.astype(bool); P = int(y.sum())
    f1_none = 2 * tp0 / max(2 * tp0 + fp0 + fn0 + P, 1)           # option: predict nothing here
    if len(s) == 0:
        return np.inf, f1_none
    order = np.argsort(-s, kind="mergesort")
    s_sorted, y_sorted = s[order], y[order]
    last = np.r_[s_sorted[1:] != s_sorted[:-1], True]
    ctp = np.cumsum(y_sorted)[last]; cfp = np.cumsum(~y_sorted)[last]
    tp, fp, fn = tp0 + ctp, fp0 + cfp, fn0 + P - ctp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    k = int(np.flatnonzero(f1 == f1.max())[-1])
    return (np.inf, f1_none) if f1_none > f1[k] else (float(s_sorted[last][k]), float(f1[k]))

def f1_thresholds(y, s):
    thr, degen = np.zeros(C), np.zeros(C, bool)
    for c in range(C):
        t, f = best_f1_threshold(y[:, c], s[:, c])
        if f == 0 or not np.isfinite(t):
            t, degen[c] = 0.5, True
        thr[c] = t
    return thr, degen

def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def val_macro_auroc(y, s):
    return float(np.nanmean(auroc_cols(y, s)))

## 4. Session L — features after every block
One pass over train, validation and test (fp16, batch 32). For each block *k* the pooled 1536-d feature is written to `features/{split}_Lkk.npy` (fp16, memory-mapped).
**QC-L3 (blocking):** the trained M09 head applied to the block-12 validation features must reproduce the saved M09 validation logits.

In [ ]:
@torch.inference_mode()
def extract_split(model, split, frame):
    done = f"{FEAT_DIR}/{split}_complete.json"
    if os.path.exists(done) and json.load(open(done)).get("n") == len(frame):
        print(f"{split}: features already extracted - skipping"); return
    D = 2 * model.backbone.config.hidden_size
    mm = {k: np.lib.format.open_memmap(f"{FEAT_DIR}/{split}_L{k:02d}.npy", mode="w+", dtype=np.float16, shape=(len(frame), D))
          for k in DEPTHS}
    i0, t0 = 0, time.time()
    dl = loader(frame, VIT_CFG, False)
    for b, (x, _) in enumerate(dl):
        with autocast():
            f = layer_features(model, x.to(DEVICE, non_blocking=True), DEPTHS)
        for k in DEPTHS:
            mm[k][i0:i0 + len(x)] = f[k].float().cpu().numpy().astype(np.float16)
        i0 += len(x)
        if b % 200 == 0:
            print(f"  {split}: {i0:,}/{len(frame):,} images, {(time.time() - t0) / 60:.1f} min", flush=True)
    assert i0 == len(frame)
    for m in mm.values():
        m.flush()
    write_json(done, dict(n=len(frame), image_index_sha256=sha256_bytes("\n".join(frame.image_index).encode()),
                          minutes=(time.time() - t0) / 60))
    print(f"{split}: {len(frame):,} images in {(time.time() - t0) / 60:.1f} min")

def feats(split, k):
    return np.load(f"{FEAT_DIR}/{split}_L{k:02d}.npy", mmap_mode="r")

if RUN_LAYERS:
    os.makedirs(FEAT_DIR, exist_ok=True)
    set_determinism()
    MODEL = load_m09(CKPT)
    with torch.inference_mode(), autocast():                             # manual block loop == model forward
        x = torch.randn(2, 3, VIT_CFG["img_size"], VIT_CFG["img_size"], device=DEVICE)
        d = (MODEL.head(layer_features(MODEL, x, [12])[12]) - MODEL(x)).abs().max().item()
    assert d < 1e-2, f"layer_features(12) differs from the model forward by {d}"
    for split in ("val", "test", "train"):                              # val first: QC-L3 needs it
        extract_split(MODEL, split, FRAMES[split])

    # ---- QC-L3: block-12 features + trained head reproduce the exported M09 logits ----
    with torch.inference_mode():
        z = MODEL.head(torch.as_tensor(np.asarray(feats("val", 12)), dtype=torch.float32, device=DEVICE)).cpu().numpy()
    z_saved = M09_NPZ["val"]["logits"][align(FRAMES["val"], M09_NPZ["val"])]
    yv_ = FRAMES["val"][CLASSES].values
    corr = min(np.corrcoef(z[:, c], z_saved[:, c])[0, 1] for c in range(C))
    auc_new, auc_saved = val_macro_auroc(yv_, z), val_macro_auroc(yv_, z_saved)
    ok = corr >= QC_L3["min_corr"] and abs(auc_new - auc_saved) <= QC_L3["max_auroc_diff"]
    qc("QC-L3 reproduces M09 logits", ok,
       f"min per-class corr {corr:.5f}, max |Δlogit| {np.abs(z - z_saved).max():.4f}, "
       f"val macro AUROC {auc_new:.5f} vs saved {auc_saved:.5f}")
    write_json(f"{OUT_DIR}/qc_session_L.json", QC)
    assert ok, "QC-L3 failed: checkpoint, preprocessing or feature path differs from the trained model - stop"

## 5. Session L — one linear probe per depth
Features are standardised with train mean/std. Head = `Linear(1536, 14)`, ASL loss, AdamW (`PROBE`), early stopping on validation macro AUROC
(patience 5, the earlier epoch wins ties) — the same rule the full models used. Validation and test logits are exported in the v2.0.0 format; **no test metric is computed here**.

In [ ]:
def train_probe(k):
    Xtr = torch.as_tensor(np.asarray(feats("train", k)), dtype=torch.float32)
    mu, sd = Xtr.mean(0), Xtr.std(0).clamp_min(1e-6)
    std = lambda a: ((torch.as_tensor(np.asarray(a), dtype=torch.float32) - mu) / sd).to(DEVICE)
    Xtr, Xva = ((Xtr - mu) / sd).to(DEVICE), std(feats("val", k))
    ytr = torch.as_tensor(FRAMES["train"][CLASSES].values, dtype=torch.float32, device=DEVICE)
    yv_ = FRAMES["val"][CLASSES].values
    seed_everything(SEED)
    head = nn.Linear(Xtr.shape[1], C).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE["lr"], weight_decay=PROBE["weight_decay"])
    loss_fn, g = AsymmetricLoss(**ASL), torch.Generator().manual_seed(SEED)
    best, best_ep, bad, state, log = -1.0, 0, 0, None, []
    for ep in range(1, PROBE["max_epochs"] + 1):
        head.train(); tl = 0.0
        for idx in torch.randperm(len(Xtr), generator=g).split(PROBE["batch_size"]):
            idx = idx.to(DEVICE)
            loss = loss_fn(head(Xtr[idx]), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tl += loss.item() * len(idx)
        head.eval()
        with torch.no_grad():
            auc = val_macro_auroc(yv_, head(Xva).cpu().numpy())
        log.append(dict(epoch=ep, train_loss=tl / len(Xtr), val_macro_auroc=auc))
        if auc > best:                                                  # strict '>' keeps the earlier epoch on ties
            best, best_ep, bad = auc, ep, 0
            state = {n: v.detach().clone() for n, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= PROBE["patience"]:
                break
    head.load_state_dict(state)
    with torch.no_grad():
        z = {"val": head(Xva).cpu().numpy(), "test": head(std(feats("test", k))).cpu().numpy()}
    return z, pd.DataFrame(log), dict(best_val_macro_auroc=best, best_epoch=best_ep, epochs_trained=len(log))

if RUN_LAYERS:
    for k in DEPTHS:
        exp_dir = f"{OUT_DIR}/{run_folder_name(depth_id(k), RUN_TAG)}"
        if os.path.exists(f"{exp_dir}/logits_test.npz"):
            print(f"depth {k:2d}: probe logits exist - skipping"); continue
        os.makedirs(exp_dir, exist_ok=True)
        t0 = time.time()
        z, log, info = train_probe(k)
        for part in ("val", "test"):
            fr = FRAMES[part]
            np.savez(f"{exp_dir}/logits_{part}.npz", image_index=fr["image_index"].to_numpy(dtype=str),
                     patient_id=fr["patient_id"].to_numpy(dtype=np.int64), logits=z[part].astype(np.float32),
                     labels=fr[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                     split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=depth_id(k), seed=SEED, eval_set=part)
        log.to_csv(f"{exp_dir}/probe_log.csv", index=False)
        write_json(f"{exp_dir}/config.json", dict(
            model_id=depth_id(k), depth=k, seed=SEED, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256,
            protocol_version=PROTOCOL_VERSION, source_model=M09_ID, source_checkpoint_sha256=EXPECTED_M09_CKPT_SHA256,
            features="block-k output -> final layernorm -> CLS ⊕ mean(patch), fp16", probe=PROBE, train_info=info,
            software=dict(torch=torch.__version__, transformers=__import__("transformers").__version__),
            gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu", debug=DEBUG))
        print(f"depth {k:2d}: val macro AUROC {info['best_val_macro_auroc']:.4f} (epoch {info['best_epoch']}/"
              f"{info['epochs_trained']}), {time.time() - t0:.0f} s", flush=True)

## 6. Session L — latency of RAD-DINO cut after *k* blocks
Same method as the v2.0.0 benchmark (forward pass on a preloaded tensor, fp16, random weights, 50 warm-up + 200 timed iterations, CUDA events,
3 repeats, median of repeat medians, batch 1 and 32). Compute ratios use the depth-12 time **from this same session** as the reference.

In [ ]:
def benchmark_depth(k):
    """Forward-pass latency per image of RAD-DINO kept to k blocks (+ pooling + head). Weights do not affect latency."""
    assert USE_AMP, "benchmark needs a GPU"
    seed_everything(SEED)
    model = RadDinoClassifier(VIT_CFG, depth=k).to(DEVICE).eval()
    res = dict(model_id=depth_id(k), depth=k, gpu=torch.cuda.get_device_name(0), torch=torch.__version__, amp_dtype="fp16",
               scope="model forward pass on a preloaded tensor, no TTA", weights="random (same architecture)",
               warmup_iters=WARMUP_ITERS, timed_iters=TIMED_ITERS, repeats=TIMING_REPEATS,
               total_params=sum(p.numel() for p in model.parameters()), source="session L benchmark")
    for bs in LATENCY_BATCH_SIZES:
        x = torch.randn(bs, 3, VIT_CFG["img_size"], VIT_CFG["img_size"], device=DEVICE)
        meds = []
        for _ in range(TIMING_REPEATS):
            with torch.inference_mode(), autocast():
                for _ in range(WARMUP_ITERS):
                    model(x)
                torch.cuda.synchronize()
                ms = []
                for _ in range(TIMED_ITERS):
                    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                    s.record(); model(x); e.record(); torch.cuda.synchronize()
                    ms.append(s.elapsed_time(e) / bs)
            meds.append(float(np.median(ms)))
        res[f"bs{bs}_median_ms_per_image"], res[f"bs{bs}_repeat_medians"] = float(np.median(meds)), meds
    write_json(f"{OUT_DIR}/timing_{depth_id(k)}{'_debug' if DEBUG else ''}.json", res)
    print(f"depth {k:2d}: " + ", ".join(f"bs{bs} {res[f'bs{bs}_median_ms_per_image']:.2f} ms" for bs in LATENCY_BATCH_SIZES), flush=True)
    del model; torch.cuda.empty_cache()

if RUN_LAYERS:
    if "MODEL" in globals():
        del MODEL; torch.cuda.empty_cache()
    for k in DEPTHS[::-1]:                                              # depth 12 (the reference) first
        if os.path.exists(f"{OUT_DIR}/timing_{depth_id(k)}{'_debug' if DEBUG else ''}.json"):
            print(f"depth {k:2d}: timing exists - skipping"); continue
        benchmark_depth(k)
    if not KEEP_FEATURES:
        shutil.rmtree(FEAT_DIR, ignore_errors=True)
    print("Session L done.")

## 7. Session E — calibration, thresholds, depth selection (validation only), test metrics
* Systems: **CNN alone** (M08), **RAD-DINO alone** (M09 with its trained head), and **Depth *k*** probes (*k* = 1 … 12).
* Per system: Platt calibration and F1 thresholds on validation, applied unchanged to test (as in v2.0.0).
* **Selection (validation only):** smallest *k* with validation macro AUROC ≥ RAD-DINO alone − `DEPTH_TOLERANCE`. If none qualifies, only the curve is reported.
* Test: 95 % CIs from `N_BOOT` patient-level resamples (same resamples for every system).
* Paired comparisons for the selected depth vs **RAD-DINO alone**, vs **CNN alone**, and vs the **depth-12 probe** (like-for-like: separates the effect of depth from the effect of a probe head), Holm-corrected per metric.

In [ ]:
if RUN_EVAL:
    SRC = {"CNN alone": (CNN_ID, DATASET_ID), "RAD-DINO alone": (M09_ID, DATASET_ID)}
    SRC.update({f"Depth {k}": (depth_id(k), RUN_TAG) for k in DEPTHS})
    NPZ, DIRS = {}, {}
    for name, (mid, tag) in SRC.items():
        d = find_run_dir(mid, tag)
        assert d, f"no logits for {mid} - attach results_s42 and the session-L output"
        DIRS[name], NPZ[name] = d, {p: np.load(f"{d}/logits_{p}.npz") for p in ("val", "test")}
    PROBES = [n for n in SRC if n.startswith("Depth")]

    # ---- QC-L4: every system scores the same images with the same labels, split and label order ----
    ref = NPZ["Depth 12"]                                               # in DEBUG the probes use a subset
    for name in NPZ:
        for p in ("val", "test"):
            pos = pd.Series(np.arange(len(NPZ[name][p]["image_index"])), index=NPZ[name][p]["image_index"].astype(str))
            idx = pos.reindex(ref[p]["image_index"].astype(str))
            assert not idx.isna().any(), f"{name} {p}: images missing"
            NPZ[name][p] = {k: (NPZ[name][p][k][idx.values.astype(int)] if NPZ[name][p][k].ndim else NPZ[name][p][k])
                            for k in NPZ[name][p].files}
    same = all(np.array_equal(NPZ[n][p]["labels"], ref[p]["labels"]) and np.array_equal(NPZ[n][p]["patient_id"], ref[p]["patient_id"])
               for n in NPZ for p in ("val", "test"))
    split_ok = {str(NPZ[n][p]["split_sha256"]) for n in NPZ for p in ("val", "test")} == {EXPECTED_SPLIT_SHA256}
    order_ok = all(json.loads(str(NPZ[n][p]["label_order"])) == CLASSES for n in NPZ for p in ("val", "test"))
    n_test_ok = DEBUG or len(ref["test"]["labels"]) == EXPECTED_COUNTS["test"]
    qc("QC-L4 identical eval sets", same and split_ok and order_ok and n_test_ok,
       f"labels/patients identical {same}, split hash {split_ok}, label order {order_ok}, full test set {n_test_ok}")
    src_ok = all(json.load(open(f"{DIRS[n]}/config.json")).get("source_checkpoint_sha256") == EXPECTED_M09_CKPT_SHA256 for n in PROBES)
    qc("QC-L1 probes built from the verified checkpoint", src_ok, f"source_checkpoint_sha256 = {EXPECTED_M09_CKPT_SHA256[:12]}… in all probe configs")
    qcl = find_file("qc_session_L.json")
    l3 = json.load(open(qcl)).get("QC-L3 reproduces M09 logits") if qcl else None
    qc("QC-L3 reproduces M09 logits", bool(l3 and l3["passed"]), l3["detail"] if l3 else "qc_session_L.json not found")
    failed = [k for k, v in QC.items() if not v["passed"] and v["severity"] == "error"]
    assert not failed, f"runs rejected: {failed}"

    yv, yt = ref["val"]["labels"].astype(bool), ref["test"]["labels"].astype(bool)
    pid_test = ref["test"]["patient_id"]

    # ---- calibration + thresholds on validation, applied unchanged to test ----
    SYSTEMS, CAL = {}, {}
    for name in SRC:
        a, b = fit_platt(NPZ[name]["val"]["logits"].astype(np.float64), yv)
        pv, pt = (expit(a * NPZ[name][p]["logits"].astype(np.float64) + b) for p in ("val", "test"))
        thr, degen = f1_thresholds(yv, pv)
        CAL[name] = dict(a=a.tolist(), b=b.tolist(), thresholds=thr.tolist(), threshold_degenerate=degen.tolist())
        SYSTEMS[name] = dict(val=(pv, pv >= thr), test=(pt, pt >= thr))
    write_json(f"{OUT_DIR}/calibration_thresholds_layers.json", CAL)

    # ---- depth selection on VALIDATION only ----
    VAL = {n: metric_values(yv, *SYSTEMS[n]["val"]) for n in SRC}
    ref_auc = VAL["RAD-DINO alone"][("auroc", "MACRO")]
    ok = [k for k in DEPTHS if VAL[f"Depth {k}"][("auroc", "MACRO")] >= ref_auc - DEPTH_TOLERANCE]
    SEL_K = min(ok) if ok else None
    SEL = f"Depth {SEL_K}" if SEL_K else None
    print(f"RAD-DINO alone val macro AUROC {ref_auc:.4f}; cut-off {ref_auc - DEPTH_TOLERANCE:.4f}")
    for k in DEPTHS:
        v = VAL[f"Depth {k}"][("auroc", "MACRO")]
        print(f"  depth {k:2d}: val macro AUROC {v:.4f} ({v - ref_auc:+.4f}){'  <- selected' if k == SEL_K else ''}")
    print(f"Selected on validation: {SEL or 'none - no depth within tolerance, curve only'}")

## 8. Session E — test metrics with patient-level bootstrap CIs, paired comparisons (T7, T8)

In [ ]:
COMPARE = [("auroc", "MACRO"), ("auprc", "MACRO"), ("f1", "MACRO"), ("f1", "MICRO"), ("precision", "MACRO"), ("recall", "MACRO")]

if RUN_EVAL:
    ORDER = ["CNN alone", "RAD-DINO alone"] + PROBES
    t0 = time.time()
    groups = list(pd.Series(np.arange(len(pid_test))).groupby(pid_test).indices.values())
    keys = list(metric_values(yt, *SYSTEMS[ORDER[0]]["test"]).keys())
    kpos = {k: i for i, k in enumerate(keys)}
    boot = np.full((N_BOOT, len(ORDER), len(keys)), np.nan)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    for b in range(N_BOOT):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        for j, name in enumerate(ORDER):
            v = metric_values(yt[idx], SYSTEMS[name]["test"][0][idx], SYSTEMS[name]["test"][1][idx])
            boot[b, j] = [v[k] for k in keys]
        if b % 100 == 0:
            print(f"  resample {b}/{N_BOOT}, {(time.time() - t0) / 60:.1f} min", flush=True)
    print(f"{N_BOOT} patient-level resamples in {(time.time() - t0) / 60:.1f} min")

    # ---- T7 results (long format, like T2) ----
    rows = []
    for j, name in enumerate(ORDER):
        for es, y in (("val", yv), ("test", yt)):
            for (m, cls), val in metric_values(y, *SYSTEMS[name][es], with_ece=True).items():
                lo_ = hi_ = np.nan
                if es == "test" and (m, cls) in kpos:
                    col = boot[:, j, kpos[(m, cls)]]; col = col[np.isfinite(col)]
                    if len(col):
                        lo_, hi_ = np.percentile(col, [2.5, 97.5])
                rows.append(dict(system=name, depth=int(name.split()[1]) if name.startswith("Depth") else (12 if name == "RAD-DINO alone" else np.nan),
                                 eval_set=es, metric=m, class_name=cls, value=val, ci_low=lo_, ci_high=hi_, direction=DIRECTION[m],
                                 threshold_strategy="none" if m in THRESHOLD_FREE else "f1_max_per_class"))
    T7 = pd.DataFrame(rows).assign(selected_on_val=lambda d: d.system == SEL, dataset_id=DATASET_ID, split_id=SPLIT_ID,
                                   split_sha256=EXPECTED_SPLIT_SHA256, seed=SEED, calibration="platt_per_class",
                                   n_bootstrap=N_BOOT, protocol_version=PROTOCOL_VERSION)
    save_table(T7, "T7_layer_results")

    # ---- T8 paired comparisons: selected depth vs 3 references (Holm per metric); every depth vs RAD-DINO (descriptive) ----
    def paired(a, b_, m, cls):
        i = kpos[(m, cls)]
        d = boot[:, ORDER.index(a), i] - boot[:, ORDER.index(b_), i]; d = d[np.isfinite(d)]
        va, vb = (metric_values(yt, *SYSTEMS[s]["test"])[(m, cls)] for s in (a, b_))
        return dict(metric=m, class_name=cls, system_a=a, system_b=b_, delta=va - vb,
                    ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                    delta_rel_pct=100 * (va - vb) / vb if vb else np.nan,
                    p_value=max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / (N_BOOT + 1)))
    comps = []
    if SEL:
        refs = [r for r in ("RAD-DINO alone", "CNN alone", "Depth 12") if r != SEL]
        for m, cls in COMPARE:
            fam = [dict(paired(SEL, r, m, cls), family="confirmatory (selected depth)") for r in refs]
            for f, adj in zip(fam, holm([f["p_value"] for f in fam])):
                f["p_holm"] = adj
                f["significant"] = bool((f["ci_low"] > 0 or f["ci_high"] < 0) and adj < ALPHA)
            comps += fam
    for k in DEPTHS:
        for m, cls in (("auroc", "MACRO"), ("f1", "MACRO")):
            comps.append(dict(paired(f"Depth {k}", "RAD-DINO alone", m, cls), family="descriptive (no significance claim)",
                              p_holm=np.nan, significant=np.nan))
    T8 = pd.DataFrame(comps)
    save_table(T8, "T8_layer_paired")
    if SEL:
        display(T8[T8.family.str.startswith("confirmatory")].round(4))

## 9. Session E — compute and the accuracy/compute curve (T9)
Compute ratio of depth *k* = latency(depth *k*) / latency(depth 12), both measured in session L. The CNN's ratio uses the v2.0.0 session-T timings (as in T6).

In [ ]:
if RUN_EVAL:
    def load_timing(model_id):
        p = find_file(f"timing_{model_id}{'_debug' if DEBUG and model_id.startswith(M09_ID + '_L') else ''}.json")
        return json.load(open(p)) if p else {}
    TIM = {k: load_timing(depth_id(k)) for k in DEPTHS}
    TIM_T = {m: load_timing(m) for m in (CNN_ID, M09_ID)}                # v2.0.0 session T
    ms = lambda t, bs: t.get(f"bs{bs}_median_ms_per_image", np.nan)
    if not all(TIM.values()):
        print("⚠ some timing_*_Lkk.json files are missing: compute columns will be NaN (run session L)")

    q = T7[(T7.eval_set == "test")].set_index(["system", "metric", "class_name"])
    qv = T7[(T7.eval_set == "val")].set_index(["system", "metric", "class_name"])
    t9 = []
    for name in ORDER:
        k = int(name.split()[1]) if name.startswith("Depth") else None
        if k:
            r1, r32 = ms(TIM[k], 1) / ms(TIM[12], 1), ms(TIM[k], 32) / ms(TIM[12], 32)
            l1, l32, src = ms(TIM[k], 1), ms(TIM[k], 32), TIM[k].get("source", "missing")
        elif name == "RAD-DINO alone":
            r1 = r32 = 1.0
            l1, l32, src = ms(TIM[12], 1), ms(TIM[12], 32), TIM[12].get("source", "missing")
        else:
            l1, l32, src = ms(TIM_T[CNN_ID], 1), ms(TIM_T[CNN_ID], 32), TIM_T[CNN_ID].get("source", "missing")
            r1, r32 = l1 / ms(TIM_T[M09_ID], 1), l32 / ms(TIM_T[M09_ID], 32)
        row = dict(system=name, depth=k if k else (12 if name == "RAD-DINO alone" else np.nan),
                   latency_bs1_ms=l1, latency_bs32_ms=l32, compute_ratio_bs1=r1, compute_ratio_bs32=r32, timing_source=src,
                   val_macro_auroc=qv.loc[(name, "auroc", "MACRO")].value, val_macro_f1=qv.loc[(name, "f1", "MACRO")].value)
        for m in ("auroc", "auprc", "f1"):
            r = q.loc[(name, m, "MACRO")]
            row.update({f"test_macro_{m}": r.value, f"test_macro_{m}_ci_low": r.ci_low, f"test_macro_{m}_ci_high": r.ci_high})
        row["selected_on_val"] = name == SEL
        t9.append(row)
    T9 = pd.DataFrame(t9)
    save_table(T9, "T9_layer_compute")
    print(f"TEST | {SPLIT_ID} ({EXPECTED_SPLIT_SHA256[:12]}) | seed {SEED} (single seed) | no TTA | Platt per class | "
          f"F1-max thresholds from val | {N_BOOT} patient-level bootstrap resamples | selected on val: {SEL}")
    display(T9.round(4))

    # ---- plot: test macro AUROC vs compute (left), validation macro AUROC vs depth with the selection cut-off (right) ----
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    pr = T9[T9.system.str.startswith("Depth")]
    ax.errorbar(pr.compute_ratio_bs32, pr.test_macro_auroc,
                yerr=[pr.test_macro_auroc - pr.test_macro_auroc_ci_low, pr.test_macro_auroc_ci_high - pr.test_macro_auroc],
                fmt="o-", capsize=3, label="RAD-DINO cut after k blocks (linear probe)")
    for _, r in pr.iterrows():
        ax.annotate(f"{int(r.depth)}", (r.compute_ratio_bs32, r.test_macro_auroc), textcoords="offset points", xytext=(4, -12), fontsize=8)
    for name, mk in (("RAD-DINO alone", "*"), ("CNN alone", "s")):
        r = T9[T9.system == name].iloc[0]
        ax.errorbar([r.compute_ratio_bs32], [r.test_macro_auroc], yerr=[[r.test_macro_auroc - r.test_macro_auroc_ci_low],
                    [r.test_macro_auroc_ci_high - r.test_macro_auroc]], fmt=mk, ms=11, capsize=3, label=name)
    if SEL:
        r = T9[T9.system == SEL].iloc[0]
        ax.scatter(r.compute_ratio_bs32, r.test_macro_auroc, s=220, facecolors="none", edgecolors="red", label=f"selected on val ({SEL})")
    ax.set_xlabel("compute vs full RAD-DINO (batch 32)"); ax.set_ylabel("test macro AUROC (95 % CI)"); ax.grid(alpha=0.3)
    ax.set_title(f"Test: accuracy vs compute | {SPLIT_ID} | seed {SEED}"); ax.legend(fontsize=8)
    ax = axes[1]
    ax.plot(pr.depth, pr.val_macro_auroc, "o-", label="probe, validation")
    ax.axhline(ref_auc, c="gray", label="RAD-DINO alone, validation")
    ax.axhline(ref_auc - DEPTH_TOLERANCE, c="red", ls="--", label=f"selection cut-off (−{DEPTH_TOLERANCE})")
    ax.set_xlabel("blocks kept (k)"); ax.set_ylabel("validation macro AUROC"); ax.set_xticks(DEPTHS); ax.grid(alpha=0.3)
    ax.set_title("Validation: depth selection"); ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/layer_depth_curve.png", dpi=150)
    plt.show()

## 10. Session E — quality control and required statements

In [ ]:
if RUN_EVAL:
    qc("QC-06 fitted on validation only", True, "calibration, thresholds, probe early stopping and depth selection use val arrays only (by construction)")
    mism = 0
    for (sys_, es, m), d in T7[T7.class_name.isin(CLASSES)].groupby(["system", "eval_set", "metric"]):
        mac = T7[(T7.system == sys_) & (T7.eval_set == es) & (T7.metric == m) & (T7.class_name == "MACRO")].value
        mism += int(len(mac) == 1 and np.isfinite(mac.item()) and abs(np.nanmean(d.value) - mac.item()) > 1e-12)
    qc("QC-07 macro = mean of per-class", mism == 0, f"{mism} mismatches")
    gpus = {t.get("gpu") for t in TIM.values()}
    qc("QC-08 timing", all(TIM.values()) and len(gpus) == 1 and not DEBUG,
       f"depth timings from {gpus}; CNN ratio from v2.0.0 session T", severity="warning")
    qc("QC-09 single seed", False, f"seed {SEED} only: CIs cover test-set sampling, not training randomness", severity="warning")
    qc("QC-10 probe is a lower bound", False, "each depth uses a linear probe on a model fine-tuned at depth 12; "
       "fine-tuning a model cut to k blocks may do better", severity="warning")
    rep = pd.DataFrame([dict(id=k, **v) for k, v in QC.items()])
    save_table(rep, "qc_report_layers")
    pd.set_option("display.max_colwidth", 200)
    display(rep)
    print("\nRequired statements for the report:")
    print(f"* Official split ({SPLIT_ID}, split_sha256 {EXPECTED_SPLIT_SHA256}); source model = seed-42 RAD-DINO "
          f"(checkpoint sha256 {EXPECTED_M09_CKPT_SHA256[:12]}…), not retrained.")
    print("* Test data used only in session E; probe early stopping, calibration, thresholds and depth selection used validation only.")
    print(f"* Single training seed ({SEED}), no TTA. Probes are linear heads on frozen features: a lower bound for a model cut to k blocks.")
    if DEBUG:
        print("* DEBUG RUN - these numbers must never be reported.")
    errors = rep[(~rep.passed) & (rep.severity == "error")]
    if len(errors):
        raise RuntimeError(f"QC errors - tables are not protocol-compliant: {errors.id.tolist()}")
    print("QC: no blocking errors.")